Real-world dataset source list can be found in: https://docs.google.com/spreadsheets/d/1yrvX6h4aO3x1vG9fkoXNRbtQuoyzLla_fXr2fF9NSH8/edit?usp=sharing

In [ ]:
import os
import re
import csv
import gzip
import shutil
import zipfile
import tarfile
import subprocess
from pathlib import Path
from urllib.parse import urljoin, urlparse

import numpy as np
import pandas as pd
import requests

In [ ]:
# =========================
# User settings
# =========================

MANIFEST_PATH = Path("/Users/ilg/Desktop/Dataset_Collection.csv")
OUT_DIR = Path("/Users/ilg/Desktop/real_world_dataset")

RAW_DIR = OUT_DIR / "raw"
REPORT_DIR = OUT_DIR / "reports"

MIN_N = 1000
MIN_P = 100

MAX_DOWNLOAD_MB = 1500
MAX_EXTRACT_MB = 3000
TABULAR_SAMPLE_ROWS = 5000

MISSING_COLUMN_THRESHOLD = 0.10
NUMERIC_PARSE_THRESHOLD = 0.90

PREFILTER_BY_MANIFEST_SIZE = True


In [ ]:
# =========================
# General helpers
# =========================

def clean_colname(x):
    return (
        str(x).strip().lower()
        .replace("?", "")
        .replace("/", "_")
        .replace(" ", "_")
        .replace("-", "_")
    )


def safe_name(x):
    x = str(x).strip()
    x = re.sub(r"[^\w\-.]+", "_", x)
    x = re.sub(r"_+", "_", x)
    return x.strip("_")[:120] or "unnamed_dataset"


def parse_number(x):
    if pd.isna(x):
        return None
    s = str(x).strip().replace(",", "")
    s = s.replace(">", "").replace("<", "")
    m = re.search(r"\d+", s)
    if not m:
        return None
    return int(m.group(0))


def get_field(row, name, default=""):
    return row[name] if name in row and not pd.isna(row[name]) else default


def ensure_dirs():
    RAW_DIR.mkdir(parents=True, exist_ok=True)
    REPORT_DIR.mkdir(parents=True, exist_ok=True)


def write_reports(rows):
    report_df = pd.DataFrame(rows)
    report_df.to_csv(REPORT_DIR / "dataset_collection_report.csv", index=False)

    if "status" in report_df.columns:
        report_df[report_df["status"].str.contains("accepted", na=False)].to_csv(
            REPORT_DIR / "accepted_datasets.csv", index=False
        )
        report_df[report_df["status"].str.contains("excluded", na=False)].to_csv(
            REPORT_DIR / "excluded_datasets.csv", index=False
        )
        report_df[report_df["status"].str.contains("manual", na=False)].to_csv(
            REPORT_DIR / "manual_review_needed.csv", index=False
        )
        report_df[report_df["status"].str.contains("failed", na=False)].to_csv(
            REPORT_DIR / "failed_download_or_parse.csv", index=False
        )


def file_size_mb(path):
    if not path.exists():
        return 0
    return path.stat().st_size / (1024 ** 2)


def url_looks_direct(url):
    path = urlparse(url).path.lower()
    return path.endswith((
        ".csv", ".tsv", ".txt", ".data", ".arff", ".mat",
        ".zip", ".gz", ".tgz", ".tar", ".tar.gz", ".npy", ".npz"
    ))


def guess_filename_from_url(url, fallback="downloaded_file"):
    parsed = urlparse(url)
    name = Path(parsed.path).name
    if not name:
        name = fallback
    return safe_name(name)


In [ ]:


# =========================
# Download helpers
# =========================

def download_url(url, dest_dir, label):
    """
    Attempts to download a direct file URL.
    Returns: (ok, local_path, reason)
    """
    dest_dir.mkdir(parents=True, exist_ok=True)

    try:
        head = requests.head(url, allow_redirects=True, timeout=20, headers={"User-Agent": "Mozilla/5.0"})
        content_length = head.headers.get("Content-Length")
        if content_length is not None:
            size_mb = int(content_length) / (1024 ** 2)
            if size_mb > MAX_DOWNLOAD_MB:
                return False, "", f"file too large by HEAD check: {size_mb:.1f} MB"

    except Exception:
        pass

    try:
        r = requests.get(url, stream=True, timeout=60, allow_redirects=True, headers={"User-Agent": "Mozilla/5.0"})
        r.raise_for_status()

        content_type = r.headers.get("Content-Type", "").lower()
        if "text/html" in content_type and not url_looks_direct(url):
            return False, "", "URL returned HTML page rather than direct downloadable data"

        filename = guess_filename_from_url(r.url, fallback=label)
        local_path = dest_dir / filename

        downloaded = 0
        max_bytes = MAX_DOWNLOAD_MB * 1024 ** 2

        with open(local_path, "wb") as f:
            for chunk in r.iter_content(chunk_size=1024 * 1024):
                if not chunk:
                    continue
                downloaded += len(chunk)
                if downloaded > max_bytes:
                    f.close()
                    local_path.unlink(missing_ok=True)
                    return False, "", f"download exceeded MAX_DOWNLOAD_MB={MAX_DOWNLOAD_MB}"
                f.write(chunk)

        return True, str(local_path), "downloaded"

    except Exception as e:
        return False, "", f"download failed: {e}"


def discover_file_links(page_url):
    """
    For UCI/Zenodo-like pages, try to discover linked data files.
    """
    try:
        r = requests.get(page_url, timeout=30, headers={"User-Agent": "Mozilla/5.0"})
        r.raise_for_status()
        html = r.text
    except Exception:
        return []

    hrefs = re.findall(r'href=["\']([^"\']+)["\']', html)
    links = []

    for href in hrefs:
        full = urljoin(page_url, href)
        low = full.lower()

        if (
            url_looks_direct(full)
            or "/static/public/" in low
            or "/files/" in low
        ):
            links.append(full)

    # Deduplicate while preserving order
    seen = set()
    out = []
    for link in links:
        if link not in seen:
            seen.add(link)
            out.append(link)

    return out[:20]


def zenodo_file_links(url):
    """
    Use the Zenodo API if the link is a Zenodo record page.
    """
    m = re.search(r"zenodo\.org/(?:record|records)/(\d+)", url)
    if not m:
        return []

    record_id = m.group(1)
    api_url = f"https://zenodo.org/api/records/{record_id}"

    try:
        r = requests.get(api_url, timeout=30)
        r.raise_for_status()
        data = r.json()
        files = data.get("files", [])
        return [f["links"]["self"] for f in files if "links" in f and "self" in f["links"]]
    except Exception:
        return []


def try_kaggle_download(url, dest_dir):
    """
    Tries Kaggle download if the kaggle CLI is installed and authenticated.
    """
    m = re.search(r"kaggle\.com/datasets/([^/]+/[^/?#]+)", url)
    if not m:
        return False, "", "not a Kaggle dataset URL or unsupported Kaggle URL"

    slug = m.group(1)

    if shutil.which("kaggle") is None:
        return False, "", "Kaggle URL found, but kaggle CLI is not installed/authenticated"

    dest_dir.mkdir(parents=True, exist_ok=True)

    try:
        cmd = ["kaggle", "datasets", "download", "-d", slug, "-p", str(dest_dir), "--unzip"]
        res = subprocess.run(cmd, capture_output=True, text=True, timeout=600)

        if res.returncode != 0:
            return False, "", f"Kaggle download failed: {res.stderr.strip()}"

        return True, str(dest_dir), "downloaded via Kaggle CLI"

    except Exception as e:
        return False, "", f"Kaggle download exception: {e}"


def try_ucimlrepo_download(url, dest_dir):
    """
    Optional UCI loader through ucimlrepo if installed.
    """
    m = re.search(r"archive\.ics\.uci\.edu/dataset/(\d+)", url)
    if not m:
        return False, "", "not a UCI dataset page with numeric ID"

    dataset_id = int(m.group(1))

    try:
        from ucimlrepo import fetch_ucirepo
    except Exception:
        return False, "", "ucimlrepo not installed"

    try:
        obj = fetch_ucirepo(id=dataset_id)
        features = obj.data.features

        if features is None:
            return False, "", "ucimlrepo returned no feature matrix"

        dest_dir.mkdir(parents=True, exist_ok=True)
        out_path = dest_dir / "uci_features.csv"
        features.to_csv(out_path, index=False)

        return True, str(out_path), "downloaded through ucimlrepo"

    except Exception as e:
        return False, "", f"ucimlrepo failed: {e}"


def collect_dataset_files(row, dataset_dir):
    """
    Returns: (download_status, downloaded_paths, reason)
    """
    url = str(get_field(row, "link", "")).strip()
    source = str(get_field(row, "source", "")).lower()

    if not url or url.lower() == "nan":
        return "manual_review_download_needed", [], "missing link"

    if "kaggle.com" in url:
        ok, path, reason = try_kaggle_download(url, dataset_dir)
        if ok:
            return "downloaded", [path], reason
        return "manual_review_download_needed", [], reason

    if "zenodo.org" in url:
        links = zenodo_file_links(url)
        if links:
            paths = []
            reasons = []
            for i, link in enumerate(links):
                ok, path, reason = download_url(link, dataset_dir, f"zenodo_file_{i}")
                reasons.append(reason)
                if ok:
                    paths.append(path)
            if paths:
                return "downloaded", paths, "; ".join(reasons)
            return "failed_download", [], "; ".join(reasons)

    if "archive.ics.uci.edu" in url:
        ok, path, reason = try_ucimlrepo_download(url, dataset_dir)
        if ok:
            return "downloaded", [path], reason

    if url_looks_direct(url):
        ok, path, reason = download_url(url, dataset_dir, "direct_download")
        if ok:
            return "downloaded", [path], reason

    discovered = discover_file_links(url)
    if discovered:
        paths = []
        reasons = []
        for i, link in enumerate(discovered):
            ok, path, reason = download_url(link, dataset_dir, f"file_{i}")
            reasons.append(f"{Path(urlparse(link).path).name}: {reason}")
            if ok:
                paths.append(path)

        if paths:
            return "downloaded", paths, "; ".join(reasons)
        return "failed_download", [], "; ".join(reasons)

    return "manual_review_download_needed", [], "could not find a direct data file link from page"



In [ ]:

# =========================
# Extraction helpers
# =========================

def extract_archives(dataset_dir):
    extracted = []

    files = list(dataset_dir.rglob("*"))
    for path in files:
        if not path.is_file():
            continue

        low = path.name.lower()

        try:
            if file_size_mb(path) > MAX_EXTRACT_MB:
                continue

            if low.endswith(".zip"):
                extract_to = dataset_dir / f"{path.stem}_extracted"
                extract_to.mkdir(exist_ok=True)
                with zipfile.ZipFile(path, "r") as z:
                    z.extractall(extract_to)
                extracted.append(str(extract_to))

            elif low.endswith((".tar", ".tar.gz", ".tgz")):
                extract_to = dataset_dir / f"{path.stem}_extracted"
                extract_to.mkdir(exist_ok=True)
                with tarfile.open(path, "r:*") as t:
                    t.extractall(extract_to)
                extracted.append(str(extract_to))

            elif low.endswith(".gz") and not low.endswith(".tar.gz"):
                out_path = path.with_suffix("")
                if not out_path.exists():
                    with gzip.open(path, "rb") as f_in, open(out_path, "wb") as f_out:
                        shutil.copyfileobj(f_in, f_out)
                extracted.append(str(out_path))

        except Exception:
            continue

    return extracted


In [ ]:


# =========================
# Screening helpers
# =========================

def is_suspicious_column_name(col):
    s = str(col).strip().lower()
    bad_terms = [
        "id", "identifier", "docid", "customer", "name", "filename",
        "file", "url", "link", "class", "label", "target", "category",
        "date", "time", "timestamp"
    ]
    return any(term == s or s.endswith("_" + term) or term in s for term in bad_terms)


def is_integer_like(values):
    values = np.asarray(values)
    values = values[np.isfinite(values)]
    if values.size == 0:
        return False
    return np.all(np.isclose(values, np.round(values)))


def screen_numeric_dataframe(df, manifest_n=None):
    """
    Estimates whether a tabular sample has enough valid numerical columns.
    """
    n_sample, p_raw = df.shape

    if p_raw < MIN_P:
        return {
            "accepted": False,
            "reason": f"too few raw columns in sample: p={p_raw}",
            "n_seen": n_sample,
            "p_seen": p_raw,
            "usable_numeric_cols": 0,
        }

    usable_cols = []
    rejected_integer_like = 0
    rejected_id_like = 0
    rejected_nonnumeric = 0
    rejected_missing = 0
    rejected_constant = 0

    for col in df.columns:
        if is_suspicious_column_name(col):
            rejected_id_like += 1
            continue

        x = pd.to_numeric(df[col], errors="coerce")
        numeric_frac = x.notna().mean()

        if numeric_frac < NUMERIC_PARSE_THRESHOLD:
            rejected_nonnumeric += 1
            continue

        missing_frac = x.isna().mean()
        if missing_frac > MISSING_COLUMN_THRESHOLD:
            rejected_missing += 1
            continue

        vals = x.dropna().to_numpy(dtype=float)
        if vals.size == 0 or np.nanstd(vals) <= 1e-12:
            rejected_constant += 1
            continue

        unique_vals = np.unique(vals)
        unique_count = len(unique_vals)
        unique_frac = unique_count / max(len(vals), 1)

        integer_like = is_integer_like(vals)

        if integer_like:
            # binary is allowed
            if unique_count <= 2:
                usable_cols.append(col)
                continue

            # likely arbitrary ID column
            if unique_frac > 0.95:
                rejected_id_like += 1
                continue

            # small integer-coded columns are often nominal categories.
            # Conservative rule: flag rather than accept.
            if unique_count <= 20:
                rejected_integer_like += 1
                continue

        usable_cols.append(col)

    enough_rows = (n_sample >= MIN_N) or (manifest_n is not None and manifest_n >= MIN_N)
    enough_cols = len(usable_cols) >= MIN_P

    if enough_rows and enough_cols:
        return {
            "accepted": True,
            "reason": "passed automatic numerical screening",
            "n_seen": n_sample,
            "p_seen": p_raw,
            "usable_numeric_cols": len(usable_cols),
            "rejected_nonnumeric": rejected_nonnumeric,
            "rejected_missing": rejected_missing,
            "rejected_constant": rejected_constant,
            "rejected_id_like": rejected_id_like,
            "rejected_ambiguous_integer_categorical": rejected_integer_like,
        }

    return {
        "accepted": False,
        "reason": (
            f"failed numerical screening: usable_numeric_cols={len(usable_cols)}, "
            f"n_sample={n_sample}, p_raw={p_raw}"
        ),
        "n_seen": n_sample,
        "p_seen": p_raw,
        "usable_numeric_cols": len(usable_cols),
        "rejected_nonnumeric": rejected_nonnumeric,
        "rejected_missing": rejected_missing,
        "rejected_constant": rejected_constant,
        "rejected_id_like": rejected_id_like,
        "rejected_ambiguous_integer_categorical": rejected_integer_like,
    }


def open_text_maybe_gzip(path):
    if str(path).lower().endswith(".gz"):
        return gzip.open(path, "rt", errors="ignore")
    return open(path, "r", errors="ignore")


def screen_docword(path):
    """
    Screens UCI bag-of-words docword files without loading the full matrix.
    Expected format:
        D
        W
        NNZ
        docID wordID count
        ...
    """
    try:
        with open_text_maybe_gzip(path) as f:
            lines = []
            for _ in range(10):
                line = f.readline()
                if not line:
                    break
                line = line.strip()
                if line:
                    lines.append(line)

        if len(lines) < 4:
            return None

        first_three_are_ints = all(re.fullmatch(r"\d+", x) for x in lines[:3])
        fourth_is_triplet = len(lines[3].split()) == 3 and all(re.fullmatch(r"\d+", x) for x in lines[3].split())

        if "docword" not in path.name.lower() and not (first_three_are_ints and fourth_is_triplet):
            return None

        D = int(lines[0])
        W = int(lines[1])
        NNZ = int(lines[2])

        accepted = D >= MIN_N and W >= MIN_P

        return {
            "accepted": accepted,
            "reason": (
                "bag-of-words sparse count matrix accepted"
                if accepted else
                f"bag-of-words matrix too small: D={D}, W={W}"
            ),
            "n_seen": D,
            "p_seen": W,
            "usable_numeric_cols": W,
            "sparse_nnz": NNZ,
            "detected_format": "docword_sparse_bag_of_words",
        }

    except Exception:
        return None


def read_tabular_sample(path):
    """
    Tries several parsing strategies and returns the sample with most columns.
    """
    candidates = []

    if path.suffix.lower() == ".arff":
        try:
            data_lines = []
            in_data = False
            with open_text_maybe_gzip(path) as f:
                for line in f:
                    if line.strip().lower() == "@data":
                        in_data = True
                        continue
                    if in_data:
                        data_lines.append(line)
                        if len(data_lines) >= TABULAR_SAMPLE_ROWS:
                            break

            if data_lines:
                from io import StringIO
                df = pd.read_csv(StringIO("".join(data_lines)), header=None)
                candidates.append(df)
        except Exception:
            pass

    attempts = [
        {"sep": None, "engine": "python"},
        {"sep": ",", "engine": "python"},
        {"sep": "\t", "engine": "python"},
        {"sep": r"\s+", "engine": "python"},
    ]

    for kwargs in attempts:
        for header in ["infer", None]:
            try:
                df = pd.read_csv(
                    path,
                    nrows=TABULAR_SAMPLE_ROWS,
                    header=header,
                    on_bad_lines="skip",
                    **kwargs,
                )
                if df.shape[1] > 1:
                    candidates.append(df)
            except Exception:
                continue

    if not candidates:
        return None

    candidates.sort(key=lambda d: (d.shape[1], d.shape[0]), reverse=True)
    return candidates[0]


def screen_mat_file(path):
    try:
        from scipy.io import loadmat
    except Exception:
        return {
            "accepted": False,
            "reason": "MAT file found, but scipy is not installed",
            "n_seen": "",
            "p_seen": "",
            "usable_numeric_cols": "",
        }

    try:
        mat = loadmat(path)
        best = None

        for key, value in mat.items():
            if key.startswith("__"):
                continue
            arr = np.asarray(value)
            if arr.ndim != 2:
                continue
            if not np.issubdtype(arr.dtype, np.number):
                continue

            n, p = arr.shape
            accepted = n >= MIN_N and p >= MIN_P

            candidate = {
                "accepted": accepted,
                "reason": f"numeric MAT matrix key={key}, shape={arr.shape}",
                "n_seen": n,
                "p_seen": p,
                "usable_numeric_cols": p,
                "mat_key": key,
            }

            if accepted:
                return candidate

            if best is None or (n * p) > (best["n_seen"] * best["p_seen"]):
                best = candidate

        if best:
            return best

        return {
            "accepted": False,
            "reason": "no 2D numeric matrix found in MAT file",
            "n_seen": "",
            "p_seen": "",
            "usable_numeric_cols": "",
        }

    except Exception as e:
        return {
            "accepted": False,
            "reason": f"MAT parse failed: {e}",
            "n_seen": "",
            "p_seen": "",
            "usable_numeric_cols": "",
        }


def screen_npy_file(path):
    try:
        arr = np.load(path, mmap_mode="r")
        if arr.ndim != 2:
            return {
                "accepted": False,
                "reason": f"NPY array is not 2D: shape={arr.shape}",
                "n_seen": "",
                "p_seen": "",
                "usable_numeric_cols": "",
            }

        n, p = arr.shape
        accepted = n >= MIN_N and p >= MIN_P and np.issubdtype(arr.dtype, np.number)

        return {
            "accepted": accepted,
            "reason": "2D numeric NPY matrix accepted" if accepted else f"NPY matrix unsuitable: shape={arr.shape}, dtype={arr.dtype}",
            "n_seen": n,
            "p_seen": p,
            "usable_numeric_cols": p if np.issubdtype(arr.dtype, np.number) else 0,
        }

    except Exception as e:
        return {
            "accepted": False,
            "reason": f"NPY parse failed: {e}",
            "n_seen": "",
            "p_seen": "",
            "usable_numeric_cols": "",
        }


def screen_file(path, manifest_n=None):
    low = path.name.lower()

    if any(x in low for x in ["readme", "license", "description", "citation", "vocab", ".names"]):
        return None

    docword_result = screen_docword(path)
    if docword_result is not None:
        return docword_result

    if low.endswith(".mat"):
        return screen_mat_file(path)

    if low.endswith(".npy"):
        return screen_npy_file(path)

    if low.endswith((".csv", ".tsv", ".txt", ".data", ".arff")):
        df = read_tabular_sample(path)
        if df is None:
            return {
                "accepted": False,
                "reason": "could not parse tabular sample",
                "n_seen": "",
                "p_seen": "",
                "usable_numeric_cols": "",
            }
        return screen_numeric_dataframe(df, manifest_n=manifest_n)

    return None


def screen_dataset_dir(dataset_dir, manifest_n=None):
    extract_archives(dataset_dir)

    candidate_files = []
    for path in dataset_dir.rglob("*"):
        if not path.is_file():
            continue
        low = path.name.lower()
        if low.endswith((
            ".csv", ".tsv", ".txt", ".data", ".arff",
            ".mat", ".npy", ".gz"
        )):
            candidate_files.append(path)

    if not candidate_files:
        return {
            "accepted": False,
            "reason": "no screenable data files found after download/extraction",
            "n_seen": "",
            "p_seen": "",
            "usable_numeric_cols": "",
            "best_file": "",
        }

    best_result = None
    best_file = ""

    for path in candidate_files:
        result = screen_file(path, manifest_n=manifest_n)
        if result is None:
            continue

        result["best_file"] = str(path)

        if result.get("accepted"):
            return result

        score = 0
        try:
            score = int(result.get("n_seen") or 0) * int(result.get("usable_numeric_cols") or 0)
        except Exception:
            score = 0

        if best_result is None:
            best_result = result
            best_file = str(path)
        else:
            try:
                old_score = int(best_result.get("n_seen") or 0) * int(best_result.get("usable_numeric_cols") or 0)
            except Exception:
                old_score = 0
            if score > old_score:
                best_result = result
                best_file = str(path)

    if best_result is None:
        return {
            "accepted": False,
            "reason": "files found, but none could be screened",
            "n_seen": "",
            "p_seen": "",
            "usable_numeric_cols": "",
            "best_file": "",
        }

    best_result["best_file"] = best_file
    return best_result



In [ ]:

# =========================
# Main pipeline
# =========================
def read_manifest_csv(path: Path) -> pd.DataFrame:
    """
    Read the dataset collection sheet robustly across common CSV encodings.
    Excel/Numbers/Google Sheets exports are not always UTF-8.
    """
    encodings_to_try = [
        "utf-8-sig",
        "utf-8",
        "gb18030",
        "gbk",
        "big5",
        "cp1252",
        "latin1",
        "utf-16",
        "utf-16le",
        "utf-16be",
    ]

    last_error = None

    for enc in encodings_to_try:
        try:
            df = pd.read_csv(
                path,
                encoding=enc,
                sep=None,
                engine="python",
                on_bad_lines="skip",
            )

            # Avoid accepting a wrongly decoded one-column mess.
            if df.shape[1] >= 3:
                print(f"Loaded manifest using encoding={enc}, shape={df.shape}")
                return df

        except Exception as e:
            last_error = e

    # Final fallback: replace undecodable characters rather than crashing.
    try:
        df = pd.read_csv(
            path,
            encoding="utf-8",
            encoding_errors="replace",
            sep=None,
            engine="python",
            on_bad_lines="skip",
        )
        print(f"Loaded manifest using utf-8 with replacement characters, shape={df.shape}")
        return df

    except Exception:
        raise RuntimeError(f"Could not read manifest file: {path}\nLast error: {last_error}")





def main():
    ensure_dirs()

    manifest = read_manifest_csv(MANIFEST_PATH)
    manifest.columns = [clean_colname(c) for c in manifest.columns]

    required = ["dataset_name", "link"]
    missing = [c for c in required if c not in manifest.columns]
    if missing:
        raise ValueError(f"Manifest is missing required columns: {missing}. Current columns: {list(manifest.columns)}")

    report_rows = []

    for idx, row in manifest.iterrows():
        raw_name = get_field(row, "dataset_name", f"dataset_{idx}")
        if not str(raw_name).strip() or str(raw_name).lower() == "nan":
            continue

        name = safe_name(raw_name)
        dataset_dir = RAW_DIR / f"{idx:03d}_{name}"
        dataset_dir.mkdir(parents=True, exist_ok=True)

        manifest_n = parse_number(get_field(row, "n", None))
        manifest_p = parse_number(get_field(row, "p", None))
        link = str(get_field(row, "link", "")).strip()
        form = str(get_field(row, "form", "")).strip()
        source = str(get_field(row, "source", "")).strip()

        print(f"\n[{idx + 1}/{len(manifest)}] {raw_name}")

        base_report = {
            "row_index": idx,
            "dataset_name": raw_name,
            "safe_dataset_name": name,
            "manifest_n": manifest_n,
            "manifest_p": manifest_p,
            "class": get_field(row, "class", ""),
            "source": source,
            "form": form,
            "link": link,
            "local_folder": str(dataset_dir),
        }

        if PREFILTER_BY_MANIFEST_SIZE:
            if manifest_n is not None and manifest_n < MIN_N:
                reason = f"excluded by manifest size: n={manifest_n} < {MIN_N}"
                print("  ->", reason)
                report_rows.append({**base_report, "status": "excluded_manifest_too_small", "reason": reason})
                write_reports(report_rows)
                continue

            if manifest_p is not None and manifest_p < MIN_P:
                reason = f"excluded by manifest size: p={manifest_p} < {MIN_P}"
                print("  ->", reason)
                report_rows.append({**base_report, "status": "excluded_manifest_too_small", "reason": reason})
                write_reports(report_rows)
                continue

        download_status, downloaded_paths, download_reason = collect_dataset_files(row, dataset_dir)

        print("  download:", download_status, "|", download_reason)

        if download_status != "downloaded":
            report_rows.append({
                **base_report,
                "status": download_status,
                "reason": download_reason,
                "downloaded_paths": "",
            })
            write_reports(report_rows)
            continue

        screen = screen_dataset_dir(dataset_dir, manifest_n=manifest_n)

        if screen.get("accepted"):
            status = "accepted_downloaded_and_screened"
        else:
            status = "excluded_after_screening"

        print("  screening:", status, "|", screen.get("reason"))

        report_rows.append({
            **base_report,
            "status": status,
            "reason": screen.get("reason", ""),
            "download_reason": download_reason,
            "downloaded_paths": " | ".join(map(str, downloaded_paths)),
            "best_file": screen.get("best_file", ""),
            "n_seen": screen.get("n_seen", ""),
            "p_seen": screen.get("p_seen", ""),
            "usable_numeric_cols": screen.get("usable_numeric_cols", ""),
            "rejected_nonnumeric": screen.get("rejected_nonnumeric", ""),
            "rejected_missing": screen.get("rejected_missing", ""),
            "rejected_constant": screen.get("rejected_constant", ""),
            "rejected_id_like": screen.get("rejected_id_like", ""),
            "rejected_ambiguous_integer_categorical": screen.get("rejected_ambiguous_integer_categorical", ""),
            "detected_format": screen.get("detected_format", ""),
            "sparse_nnz": screen.get("sparse_nnz", ""),
            "mat_key": screen.get("mat_key", ""),
        })

        write_reports(report_rows)

    write_reports(report_rows)

    print("\nDone.")
    print(f"Raw downloaded files: {RAW_DIR}")
    print(f"Reports: {REPORT_DIR}")
    print(f"Main report: {REPORT_DIR / 'dataset_collection_report.csv'}")
    print(f"Excluded report: {REPORT_DIR / 'excluded_datasets.csv'}")
    print(f"Manual review report: {REPORT_DIR / 'manual_review_needed.csv'}")


main()